In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/preprocess/y_test.csv
/kaggle/input/preprocess/y_train.csv
/kaggle/input/preprocess/X_test_scaled.csv
/kaggle/input/preprocess/scaler.joblib
/kaggle/input/preprocess/__results__.html
/kaggle/input/preprocess/rewards_test_unscaled.csv
/kaggle/input/preprocess/__notebook__.ipynb
/kaggle/input/preprocess/X_train_scaled.csv
/kaggle/input/preprocess/__output__.json
/kaggle/input/preprocess/rewards_train_unscaled.csv
/kaggle/input/preprocess/custom.css


In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
import os
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')
print(f"TensorFlow Version: {tf.__version__}")

# ===================================================================
# STEP 1: Load Preprocessed Data
# ===================================================================

# This code block automatically finds your input files from Notebook 1



print("Loading data...")
X_train = pd.read_csv('/kaggle/input/preprocess/X_train_scaled.csv')
X_test = pd.read_csv('/kaggle/input/preprocess/X_test_scaled.csv')
y_train = pd.read_csv('/kaggle/input/preprocess/y_train.csv').values.ravel() # .ravel() converts to 1D array
y_test = pd.read_csv('/kaggle/input/preprocess/y_test.csv').values.ravel()

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train count (1s): {np.sum(y_train)} / {len(y_train)}")

# Get the number of features (input dimensions for the model)
n_features = X_train.shape[1]

# ===================================================================
# STEP 2: Build the Deep Learning Model (MLP)
# ===================================================================

model = Sequential([
    # Input layer: Must match the number of features
    Dense(64, activation='relu', input_shape=(n_features,)),
    Dropout(0.3), # Dropout helps prevent overfitting
    
    # Hidden layer
    Dense(32, activation='relu'),
    Dropout(0.2),
    
    # Output layer: 1 neuron with 'sigmoid' for a 0-to-1 probability
    Dense(1, activation='sigmoid')
])

model.summary()

# ===================================================================
# STEP 3: Compile and Train the Model
# ===================================================================

# We compile the model with a loss function for binary problems,
# an optimizer, and the AUC metric to monitor.
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=[tf.keras.metrics.AUC(name='auc')])

print("\nStarting model training...")

# --- Class Weighting ---
# Our data is imbalanced (way more 0s than 1s).
# These weights tell the model to "pay more attention" to the
# minority class (1s, the defaults) so it doesn't just ignore them.
total = len(y_train)
pos = np.sum(y_train == 1)
neg = np.sum(y_train == 0)

weight_for_0 = (1 / neg) * (total / 2.0)
weight_for_1 = (1 / pos) * (total / 2.0)
class_weight = {0: weight_for_0, 1: weight_for_1}

print(f"Class Weights -> 0 (Paid): {weight_for_0:.2f}, 1 (Default): {weight_for_1:.2f}")

# Train the model
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=20,          # 20 passes over the data is a good start
    batch_size=256,
    class_weight=class_weight, # Apply the weights
    verbose=1
)

print("Training complete.")

# ===================================================================
# STEP 4: Evaluate the Model (Task 2 Requirement)
# ===================================================================

print("\nEvaluating model on test set...")

# Get predicted probabilities (a float between 0.0 and 1.0)
y_pred_proba = model.predict(X_test).ravel()

# Get binary predictions (0 or 1) using a 0.5 threshold
y_pred_binary = (y_pred_proba > 0.5).astype(int)

# 1. Calculate AUC
# We use the probabilities (y_pred_proba) for AUC
auc = roc_auc_score(y_test, y_pred_proba)

# 2. Calculate F1-Score
# We use the binary predictions (y_pred_binary) for F1
f1 = f1_score(y_test, y_pred_binary)

# 3. Show Confusion Matrix for context
cm = confusion_matrix(y_test, y_pred_binary)

print("\n\n" + "="*35)
print("--- [SUCCESS] DL Model Evaluation ---")
print("="*35)
print(f"  Test Set AUC:      {auc:.4f}")
print(f"  Test Set F1-Score: {f1:.4f}")
print("\n  Confusion Matrix (Test Set):")
print("  (Rows: Actual, Cols: Predicted)")
print(f"         [0]    [1]")
print(f"  [0]  {cm[0][0]:>6} {cm[0][1]:>6}  (Actual Paid)")
print(f"  [1]  {cm[1][0]:>6} {cm[1][1]:>6}  (Actual Default)")
print("="*35)

2025-10-30 06:19:54.160295: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761805194.377069      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761805194.428459      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow Version: 2.18.0
Loading data...
X_train shape: (141612, 36)
X_test shape:  (35404, 36)
y_train count (1s): 28819 / 141612


I0000 00:00:1761805208.888478      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1761805208.889297      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         2,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...


Starting model training...
Class Weights -> 0 (Paid): 0.63, 1 (Default): 2.46
Epoch 1/20


I0000 00:00:1761805212.186603      60 service.cc:148] XLA service 0x7fd44c00b6c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1761805212.187167      60 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1761805212.187188      60 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1761805212.464327      60 cuda_dnn.cc:529] Loaded cuDNN version 90300


 70/554 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - auc: 0.6022 - loss: 0.6861

I0000 00:00:1761805214.226488      60 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


554/554 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - auc: 0.6864 - loss: 0.6391 - val_auc: 0.7395 - val_loss: 0.6027
Epoch 2/20
554/554 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.7321 - loss: 0.6061 - val_auc: 0.7411 - val_loss: 0.5972
Epoch 3/20
554/554 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.7346 - loss: 0.6035 - val_auc: 0.7421 - val_loss: 0.6000
Epoch 4/20
554/554 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.7375 - loss: 0.6013 - val_auc: 0.7426 - val_loss: 0.5966
Epoch 5/20
554/554 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.7405 - loss: 0.5998 - val_auc: 0.7425 - val_loss: 0.5921
Epoch 6/20
554/554 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.7404 - loss: 0.6017 - val_auc: 0.7433 - val_loss: 0.5984
Epoch 7/20
554/554 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.7436 - loss: 0.5974 - val_auc: 0.7438 - val_loss: 0.5862
Epoch 8/20
554/554 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.7416 - loss: 0.6012 - val_auc: 0.7438 - val_loss: 0.5851
Epoch 9/20
554/554 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.7413 -